# Step 5b — Replicating H1 in GSE184950 (Wang et al. 2024)

**Project:** pd-da-aging · **Protocol:** fixed in docs/ANALYSIS_PLAN.md (27 Nov 2026) before any GSE184950 scores were computed.

Run the cells **one at a time, top to bottom**, and stop at each 🛑. **Needs:** `pd-da-aging_step5` (from step 5a) and `pd-da-aging_step3` (gene lists and the labelled GSE178265 neurons) in *My Drive*. Runtime: CPU.

## Cell 1 — Connect Google Drive and install tools
Click **Allow**. The red pip warning about pandas is harmless.

In [ ]:
# Connect Google Drive and install tools (2-3 minutes)
from google.colab import drive
drive.mount("/content/drive")
!pip install -q scanpy harmonypy igraph leidenalg statsmodels scikit-learn


## Cell 2 — Load the GSE184950 dopamine neurons saved in step 5a
Applies the same quality filter as step 2 and uses the same gene lists as step 3.

In [ ]:
# Set up, record versions, load the GSE184950 dopamine-neuron candidates saved in step 5a (1 minute)
import os, sys, glob, json, datetime, platform, warnings
import numpy as np, pandas as pd, scipy, scanpy as sc, anndata as ad, statsmodels, sklearn
import statsmodels.formula.api as smf
from scipy import stats, sparse
import matplotlib
import matplotlib.pyplot as plt
import harmonypy, igraph, leidenalg, numba
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*(converge|singular|boundary|Hessian).*")
sc.settings.n_jobs = max(1, int(numba.config.NUMBA_NUM_THREADS))

DRIVE = "/content/drive/MyDrive"
IN = f"{DRIVE}/pd-da-aging_step5"
OUT = f"{DRIVE}/pd-da-aging_step5"
for sub in ["/figures", "/tables"]:
    os.makedirs(OUT + sub, exist_ok=True)
TODAY = datetime.date.today().isoformat()
versions = {"date": TODAY, "python": sys.version.split()[0], "platform": platform.platform()}
for m in [np, pd, scipy, sc, ad, statsmodels, sklearn, matplotlib, harmonypy, igraph, leidenalg]:
    versions[m.__name__] = getattr(m, "__version__", "unknown")
with open(f"{OUT}/software_versions_step5b.txt", "w") as fh:
    fh.write("\n".join(f"{k}: {v}" for k, v in versions.items()) + "\n")

parts = [ad.read_h5ad(p) for p in sorted(glob.glob(f"{IN}/da_candidates/*_DA.h5ad"))]
da = ad.concat(parts, join="inner")
meta = pd.read_csv(f"{IN}/tables/GSE184950_samples.csv").set_index("gsm")
status_map = {"Unaffected Control": "Ctrl", "Parkinson's Disease": "PD", "Parkinson's Disease Dementia": "PDD"}
da.obs["sample"] = da.obs["gsm"].astype(str)
da.obs["donor"] = da.obs["sample"].map(meta["brain bank donor id"]).astype(str)
da.obs["status"] = da.obs["sample"].map(meta["disease state"]).map(status_map).astype(str)
da.obs["disease"] = np.where(da.obs["status"] == "Ctrl", "Ctrl", "Disease")
da.obs["sex"] = da.obs["sample"].map(meta["gender"]).astype(str)
da.obs["age"] = pd.to_numeric(da.obs["sample"].map(meta["age"]), errors="coerce")
mt = da.var_names.str.upper().str.startswith("MT-")
da.obs["pct_mito"] = 100 * np.asarray(da.X[:, mt].sum(1)).ravel() / np.maximum(da.obs["total_counts_raw"].to_numpy(), 1)
n0 = da.n_obs
da = da[(da.obs["n_genes_raw"] >= 500) & (da.obs["pct_mito"] < 5)].copy()      # same QC as step 2
da.layers["counts"] = da.X.copy()
print(f"{n0:,} candidates loaded; {da.n_obs:,} pass QC (>= 500 genes, < 5% mitochondrial) x {da.n_vars:,} genes")
print(da.obs.groupby(["status", "donor"]).size().rename("neurons").to_string())

# Gene lists: the SAME files used in step 3 (no new download)
gs_file = sorted(glob.glob(f"{DRIVE}/pd-da-aging_step3/gene_sets/gene_sets_*.json"))[-1]
gene_sets = json.load(open(gs_file))["gene_sets"]
kamath_panel = gene_sets["kamath_marker_panel"]
kamath_subtypes = ['SOX6_AGTR1', 'SOX6_PART1', 'SOX6_DDT', 'SOX6_GFRA2', 'CALB1_CALCR', 'CALB1_CRYM_CCDC68',
                   'CALB1_PPP1R17', 'CALB1_RBP4', 'CALB1_GEM', 'CALB1_TRHR']    # Kamath et al. 2022, Figure 1E order
print(f"\nGene lists from {os.path.basename(gs_file)} (same as step 3)")


## Cell 3 — Re-cluster the dopamine neurons (as in step 3)
Senescence and aging genes are kept out of the clustering.

In [ ]:
# Re-cluster the dopamine neurons, readout genes excluded, exactly as in step 3 (2-5 minutes)
readout_genes = set().union(*[set(gene_sets[k]) for k in ["SenMayo", "CellAge_induces", "LU_AGING_BRAIN_UP", "CDKN1A", "CDKN2A"] if gene_sets.get(k)])
da.X = da.layers["counts"].copy()
sc.pp.normalize_total(da, target_sum=1e4)
sc.pp.log1p(da)
da.layers["lognorm"] = da.X.copy()
sc.pp.highly_variable_genes(da, n_top_genes=2000)
dropped = int((da.var["highly_variable"] & da.var_names.isin(readout_genes)).sum())
da.var.loc[da.var_names.isin(readout_genes), "highly_variable"] = False
print(f"Variable genes used for clustering: {int(da.var['highly_variable'].sum())} ({dropped} readout genes removed)")
sc.tl.pca(da, n_comps=30, mask_var="highly_variable")
ho = harmonypy.run_harmony(da.obsm["X_pca"], da.obs, "sample", max_iter_harmony=20)
Z = np.asarray(ho.Z_corr)
da.obsm["X_pca_harmony"] = Z if Z.shape[0] == da.n_obs else Z.T
sc.pp.neighbors(da, use_rep="X_pca_harmony", n_neighbors=20)
RESOLUTION = 1.0
sc.tl.leiden(da, resolution=RESOLUTION, flavor="igraph", n_iterations=2, directed=False, key_added="subcluster")
sc.tl.umap(da)
print(f"{da.obs['subcluster'].nunique()} sub-clusters among {da.n_obs:,} dopamine neurons (resolution {RESOLUTION})")


## Cell 4 — Label the subtypes
Same Kamath-marker method as step 3, checked against labels transferred from the 16,755 labelled GSE178265 neurons.
### 🛑 CHECKPOINT — send Claude screenshots of the table, counts, agreement table, dot plot and UMAP. Do not run Cell 5 until Claude confirms.

In [ ]:
# Name each sub-cluster by Kamath's markers (as in step 3), and check against labels transferred from GSE178265 (1-2 minutes)
subtype_genes = {s: s.split("_") for s in kamath_subtypes}
panel = [g for g in dict.fromkeys(kamath_panel + [g for gs in subtype_genes.values() for g in gs]) if g in da.var_names]
expr = sc.get.obs_df(da, keys=panel, layer="lognorm")
cl = da.obs["subcluster"].astype(str)
means = expr.groupby(cl.to_numpy()).mean()
zmeans = (means - means.mean()) / means.std(ddof=0).replace(0, 1)
rows = []
for c in zmeans.index:
    z = zmeans.loc[c]
    family = "SOX6" if z.get("SOX6", 0) - z.get("CALB1", 0) > 0 else "CALB1"
    cands = {s: np.mean([z[g] for g in gs[1:] if g in z.index]) for s, gs in subtype_genes.items()
             if gs[0] == family and any(g in z.index for g in gs[1:])}
    ranked = sorted(cands.items(), key=lambda kv: -kv[1])
    best, second = ranked[0], (ranked[1] if len(ranked) > 1 else (None, np.nan))
    rows.append({"subcluster": c, "cells": int((cl == c).sum()), "family": family,
                 "SOX6_minus_CALB1": round(z.get("SOX6", 0) - z.get("CALB1", 0), 2),
                 "subtype": best[0], "score": round(best[1], 2), "runner_up": second[0],
                 "runner_up_score": round(second[1], 2), "margin": round(best[1] - second[1], 2)})
labels = pd.DataFrame(rows).set_index("subcluster").sort_values(["family", "subtype"])
da.obs["subtype"] = cl.map(labels["subtype"]).astype(str)
grp = lambda s: np.where(s == "SOX6_AGTR1", "vulnerable", np.where(pd.Series(s).str.startswith("CALB1"), "resilient", "other_SOX6"))
da.obs["group"] = grp(da.obs["subtype"].to_numpy())

# Label transfer: a classifier trained on the 16,755 labelled GSE178265 neurons from step 3
from sklearn.linear_model import LogisticRegression
ref = ad.read_h5ad(f"{DRIVE}/pd-da-aging_step3/GSE178265_DA_neurons_step3.h5ad")
sc.pp.normalize_total(ref, target_sum=1e4); sc.pp.log1p(ref)
sc.pp.highly_variable_genes(ref, n_top_genes=2000)
feat = [g for g in ref.var_names[ref.var["highly_variable"]] if g in da.var_names and g not in readout_genes]
Xr = ref[:, feat].X; Xq = da[:, feat].layers["lognorm"]
clf = LogisticRegression(C=0.1, max_iter=2000).fit(Xr, ref.obs["subtype"].astype(str).to_numpy())
da.obs["subtype_transfer"] = clf.predict(Xq)
da.obs["group_transfer"] = grp(da.obs["subtype_transfer"].to_numpy())
del ref
agree = (da.obs["group"] == da.obs["group_transfer"]).mean()

pd.set_option("display.width", 250)
print(labels.to_string())
print("\nNeurons per subtype (cluster labels):")
print(da.obs["subtype"].value_counts().to_string())
print(f"\nLabel transfer from GSE178265 used {len(feat)} genes; agreement on vulnerable/resilient/other group: {agree:.1%}")
print(pd.crosstab(da.obs["group"], da.obs["group_transfer"], rownames=["cluster label"], colnames=["transferred"]).to_string())
labels.to_csv(f"{OUT}/tables/subcluster_subtype_labels.csv")
order = [s for s in kamath_subtypes if s in set(da.obs["subtype"])]
sc.pl.dotplot(da, var_names=[g for g in kamath_panel if g in da.var_names], groupby="subtype",
              categories_order=order, layer="lognorm", standard_scale="var", show=False)
plt.savefig(f"{OUT}/figures/fig_subtype_marker_dotplot.png", dpi=200, bbox_inches="tight"); plt.show()
sc.pl.umap(da, color=["subtype", "donor", "SOX6", "CALB1", "AGTR1"], layer="lognorm", ncols=2, frameon=False, show=False)
plt.savefig(f"{OUT}/figures/fig_subtype_umap.png", dpi=150, bbox_inches="tight"); plt.show()


## Cell 5 — Score every neuron (same gene lists as step 3)

In [ ]:
# Score every neuron with the same gene lists as step 3 (seconds)
marker_genes = set(kamath_panel) | {g for gs in subtype_genes.values() for g in gs}
score_sets = {}
for name in ["SenMayo", "CellAge_induces", "LU_AGING_BRAIN_UP"]:
    gs = gene_sets.get(name)
    if not gs:
        continue
    kept = [g for g in gs if g in da.var_names and g not in marker_genes]
    score_sets[name] = kept
    sc.tl.score_genes(da, kept, score_name=name, use_raw=False, random_state=0)
    print(f"{name}: {len(kept)} genes used")
READOUTS = list(score_sets)


## Cell 6 — H1 replication
Vulnerable vs resilient neurons inside each control donor; direction is the pre-specified replication criterion.
### 🛑 Send Claude a screenshot of the result and the plot.

In [ ]:
# H1 replication: vulnerable vs resilient within each CONTROL donor (seconds)
def donor_pairs(obs, readout, min_cells, keep=None, gcol="group"):
    d = obs if keep is None else obs[keep]
    d = d[d[gcol].isin(["vulnerable", "resilient"])]
    n = d.groupby(["donor", gcol]).size().unstack(fill_value=0)
    for g in ["vulnerable", "resilient"]:
        if g not in n: n[g] = 0
    m = d.groupby(["donor", gcol])[readout].mean().unstack()
    ok = n.index[(n["vulnerable"] >= min_cells) & (n["resilient"] >= min_cells)]
    out = m.loc[ok, ["vulnerable", "resilient"]].copy()
    out["diff"] = out["vulnerable"] - out["resilient"]
    out["n_vulnerable"], out["n_resilient"] = n.loc[ok, "vulnerable"], n.loc[ok, "resilient"]
    return out.join(d.groupby("donor")[["status", "sex", "age"]].first())

def paired_test(p):
    k = len(p)
    res = {"donors": k, "donors_negative": int((p["diff"] < 0).sum()),
           "sign_test_p": stats.binomtest(int((p["diff"] < 0).sum()), k, 0.5).pvalue if k else np.nan}
    if k < 3:
        return {**res, "mean_diff": p["diff"].mean() if k else np.nan, "ci_low": np.nan, "ci_high": np.nan,
                "p_ttest": np.nan, "cohens_dz": np.nan, "p_wilcoxon": np.nan}
    t = stats.ttest_1samp(p["diff"], 0.0); se = p["diff"].std(ddof=1) / np.sqrt(k); q = stats.t.ppf(0.975, k - 1)
    return {**res, "mean_diff": p["diff"].mean(), "ci_low": p["diff"].mean() - q * se, "ci_high": p["diff"].mean() + q * se,
            "p_ttest": t.pvalue, "cohens_dz": p["diff"].mean() / p["diff"].std(ddof=1),
            "p_wilcoxon": stats.wilcoxon(p["diff"]).pvalue if (p["diff"] != 0).any() else np.nan}

ctrl = da.obs["status"] == "Ctrl"
MIN_CELLS = 20
pairs = donor_pairs(da.obs, "SenMayo", MIN_CELLS, keep=ctrl)
if len(pairs) < 5:
    print(f"Only {len(pairs)} control donors have >= 20 neurons of each group; using the pre-specified fallback of 10.")
    MIN_CELLS = 10
    pairs = donor_pairs(da.obs, "SenMayo", MIN_CELLS, keep=ctrl)
print(f"Control donors included: {len(pairs)} (minimum {MIN_CELLS} neurons per group)\n")
print(pairs.round(4).to_string())
h1 = paired_test(pairs)
verdict = ("REPLICATED (same direction as GSE178265)" if h1["donors"] > 0 and h1["mean_diff"] < 0 and h1["donors_negative"] > h1["donors"] / 2
           else "NOT REPLICATED in direction")
print("\nH1 REPLICATION (SenMayo, controls, vulnerable minus resilient):")
for k, v in h1.items():
    print(f"  {k}: {v:.4g}" if isinstance(v, (float, np.floating)) else f"  {k}: {v}")
print(f"  verdict (pre-specified direction criterion): {verdict}")
print("  For comparison, GSE178265: mean_diff -0.0217 (95% CI -0.0239 to -0.0195), 7 of 7 donors negative")
pairs.to_csv(f"{OUT}/tables/H1_replication_donor_pairs_SenMayo.csv")
pd.Series({**h1, "verdict": verdict, "min_cells": MIN_CELLS}).to_csv(f"{OUT}/tables/H1_replication_result_SenMayo.csv")

fig, ax = plt.subplots(figsize=(4, 4.5))
for dn, r in pairs.iterrows():
    ax.plot([0, 1], [r["resilient"], r["vulnerable"]], "-o", color="grey", alpha=0.8)
    ax.annotate(dn, (1.03, r["vulnerable"]), fontsize=7, va="center")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Resilient\n(CALB1+)", "Vulnerable\n(SOX6_AGTR1)"])
ax.set_ylabel("SenMayo score (donor mean)"); ax.set_xlim(-0.3, 1.5)
ax.set_title(f"Replication, GSE184950 control donors (n={len(pairs)})\n{h1['donors_negative']} of {len(pairs)} lower in vulnerable", fontsize=9)
plt.savefig(f"{OUT}/figures/fig_H1_replication_SenMayo_paired.png", dpi=200, bbox_inches="tight"); plt.show()


## Cell 7 — Sensitivity checks, other readouts, and NRG1/SERPINE2
### 🛑 Send Claude a screenshot.

In [ ]:
# Sensitivity checks, other readouts, and the step 4 genes NRG1 and SERPINE2 (1 minute)
rows = []
def add(label, obs=None, readout="SenMayo", keep=ctrl, gcol="group", min_cells=MIN_CELLS):
    obs = da.obs if obs is None else obs
    r = paired_test(donor_pairs(obs, readout, min_cells, keep=keep, gcol=gcol)); r["analysis"] = label; rows.append(r)
add("Primary replication")
add("Labels transferred from GSE178265", da.obs, gcol="group_transfer")
d = da.obs.copy()
Xd = np.column_stack([np.ones(len(d)), np.log1p(d["total_counts_raw"]), np.log1p(d["n_genes_raw"])])
beta, *_ = np.linalg.lstsq(Xd, d["SenMayo"].to_numpy(), rcond=None)
d["SenMayo_depth_adj"] = d["SenMayo"].to_numpy() - Xd @ beta
add("Depth-adjusted score", d, readout="SenMayo_depth_adj")
for dn in pairs.index:
    add(f"Leave out donor {dn}", da.obs, keep=ctrl & (da.obs["donor"] != dn))
for ro in READOUTS[1:]:
    add(f"Secondary readout: {ro}", da.obs, readout=ro)
sens = pd.DataFrame(rows).set_index("analysis")
print(sens[["donors", "donors_negative", "mean_diff", "ci_low", "ci_high", "p_ttest", "p_wilcoxon", "sign_test_p"]].round(4).to_string())
sens.to_csv(f"{OUT}/tables/H1_replication_sensitivity.csv")

# Step 4 drivers: pseudobulk log2 fold change (vulnerable / resilient) per qualifying control donor
cnt = da.layers["counts"]
out = []
for dn in pairs.index:
    r = {"donor": dn}
    for g in ["vulnerable", "resilient"]:
        idx = np.flatnonzero(ctrl.to_numpy() & (da.obs["donor"] == dn).to_numpy() & (da.obs["group"] == g).to_numpy())
        pb = np.asarray(cnt[idx].sum(0)).ravel()
        r[g] = np.log2(pb / pb.sum() * 1e6 + 1)
    row = {"donor": dn}
    for gene in ["NRG1", "SERPINE2", "TIMP2", "PAPPA", "CTSB", "IGFBP4", "BMP6", "KITLG"]:
        if gene in da.var_names:
            j = da.var_names.get_loc(gene); row[gene] = r["vulnerable"][j] - r["resilient"][j]
    out.append(row)
drivers = pd.DataFrame(out).set_index("donor")
print("\nlog2 fold change (vulnerable / resilient), pseudobulk, per control donor. In GSE178265:"
      " NRG1 -1.50, SERPINE2 -2.20, TIMP2 -1.11, PAPPA -0.80, CTSB -0.62, IGFBP4 -2.44, BMP6 +1.07, KITLG +1.83")
print(drivers.round(2).to_string())
drivers.to_csv(f"{OUT}/tables/step4_drivers_replication.csv")


## Cell 8 — Save and pack for GitHub
Downloads `pd-da-aging_step5_for_github.zip`. Attach it in the chat with Claude.

In [ ]:
# Save everything and pack figures + tables for GitHub (1 minute)
keep_cols = ["sample", "donor", "status", "sex", "age", "n_genes_raw", "total_counts_raw", "pct_mito", "subcluster",
             "subtype", "group", "subtype_transfer", "group_transfer"] + READOUTS
da.obs[keep_cols].to_csv(f"{OUT}/tables/per_neuron_scores_GSE184950.csv")
o = da.copy(); o.X = o.layers["counts"]
for l in ["counts", "lognorm"]:
    del o.layers[l]
for k in ["X_pca", "X_pca_harmony"]:
    o.obsm.pop(k, None)
o.write_h5ad(f"{OUT}/GSE184950_DA_neurons_step5.h5ad", compression="gzip")
import zipfile
ZIP = f"{DRIVE}/pd-da-aging_step5_for_github.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(f"{OUT}/figures/*") + glob.glob(f"{OUT}/tables/*") + glob.glob(f"{OUT}/software_versions*")):
        if os.path.getsize(p) <= (5e6 if p.endswith(".png") else 2e6):
            z.write(p, "results/step5/" + os.path.relpath(p, OUT))
for root, _, files in os.walk(OUT):
    if "da_candidates" in root:
        continue
    for f in sorted(files):
        p = os.path.join(root, f); print(f"{os.path.getsize(p) / 1e6:8.2f} MB  {p.replace(DRIVE + '/', '')}")
print(f"\nPacked for GitHub: My Drive/{os.path.basename(ZIP)}")
from google.colab import files
files.download(ZIP)
